# Brazilian Oil & Gas Production — walkthrough

Thin notebook: every computation is a function in `src/oilgas/`. Run the pipeline first (`python -m oilgas.pipeline`). Data: ANP, snapshot 2026-11-18; last month Sep 2026 (provisional).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd
from oilgas import eda, stats_analysis
from oilgas.config import PROCESSED_DIR
f, u = eda.load()
f.shape, u.shape

((75855, 30), (12852, 23))

## 1. Descriptive statistics and long-run trend

In [2]:
eda.descriptive(f).round(4)

,series,unit,first_month,last_month,n_months,mean_daily,median_daily,std_daily,min_daily,max_daily,peak_month,last_daily,cagr_2000_2025,cagr_2015_2025,cagr_2020_2025
0,oil,m3/d,1997-01-01,2026-09-01,357,346425.3847,326329.6736,132233.9928,117075.1935,748985.2968,2026-09-01,748985.2968,0.0458,0.0446,0.0510
1,gas,1000 m3/d,1997-01-01,2026-09-01,357,84216.5177,67103.5383,46953.0647,24890.9032,223878.2611,2026-09-01,223878.2611,0.0660,0.0642,0.0706
2,lgn,m3/d,1997-01-01,2026-09-01,357,12981.2955,13599.2581,4624.2874,3907.2903,31133.6066,1998-07-01,15527.3951,0.0341,-0.0078,-0.0369


In [3]:
eda.log_linear_trend(f).round(4)

,series,window,n_years,annual_growth,r2
0,oil,2000-2009,10,0.0493,0.9665
1,oil,2010-2016,7,0.0357,0.7456
2,oil,2017-2025,9,0.0454,0.9277
3,oil,2000-2025,26,0.0412,0.9856
4,gas,2000-2009,10,0.0522,0.9499
5,gas,2010-2016,7,0.0921,0.9864
6,gas,2017-2025,9,0.0579,0.9621
7,gas,2000-2025,26,0.0658,0.9905


## 2. Annual production, onshore vs offshore

In [4]:
a = eda.annual_table(f)
a[(a.series != 'lgn') & (a.year >= 2015)].pivot(index='year', columns='series', values=['daily_avg', 'offshore_share']).round(3)

daily_avg             offshore_share       
series         gas         oil            gas    oil
year                                                
2015     96236.841  387522.049          0.761  0.934
2016    103525.820  399089.008          0.770  0.940
2017    109910.692  416820.166          0.804  0.952
2018    111937.554  411240.236          0.804  0.957
2019    122532.142  443218.063          0.814  0.963
2020    127457.672  467414.761          0.839  0.968
2021    133765.279  461877.317          0.830  0.970
2022    137912.551  480381.106          0.853  0.974
2023    149800.371  540814.800          0.858  0.977
2024    153195.597  533872.314          0.848  0.975
2025    179237.138  599388.883          0.864  0.976

## 3. Gas utilisation (national)

In [5]:
g = eda.gas_utilization_annual(u)
g[g.level == 'national'].set_index('year')[['reinjection_rate', 'flaring_loss_rate', 'own_use_rate', 'available_rate']].tail(6).round(4)

,reinjection_rate,flaring_loss_rate,own_use_rate,available_rate
year,,,,
2020,0.4290,0.0264,0.1147,0.4298
2021,0.4549,0.0253,0.1070,0.4129
2022,0.4960,0.0252,0.1053,0.3734
2023,0.5261,0.0258,0.1042,0.3481
2024,0.5431,0.0283,0.1033,0.3301
2025,0.5440,0.0283,0.0983,0.3339


## 4. Structural breaks (BIC-selected)

In [6]:
breaks, bic = stats_analysis.structural_breaks(f)
breaks.round(4)

,series,start,end,n_months,annualised_growth,mean_daily_avg,metric
0,oil,1997-01,2000-08,44,0.1261,159957.6975,ln(monthly daily average)
1,oil,2000-09,2012-03,139,0.0466,270936.1002,ln(monthly daily average)
2,oil,2012-04,2016-06,51,0.0647,352502.3494,ln(monthly daily average)
3,oil,2016-07,2019-06,36,-0.0074,414703.5269,ln(monthly daily average)
4,oil,2019-07,2022-06,36,-0.0089,466464.5652,ln(monthly daily average)
5,oil,2022-07,2026-09,51,0.0874,574037.9333,ln(monthly daily average)
6,gas,1997-01,2001-11,59,0.0931,32563.8238,ln(monthly daily average)
7,gas,2001-12,2007-11,72,0.0328,46323.5181,ln(monthly daily average)
8,gas,2007-12,2012-05,54,0.0472,61845.5151,ln(monthly daily average)
9,gas,2012-06,2016-06,49,0.0976,86165.7291,ln(monthly daily average)


## 5. Charts

![production](../assets/01_national_production.png)

![gas](../assets/03_gas_utilization.png)